# 01 數據檢查

目標：確認下載的數據能用，並且搞清楚它的「坑」。這些坑都是面試會追問的地方。

做法：每個任務**先自己寫**，卡住了再看提示；還是不會就把你的代碼和報錯發給我。做完後把結論寫進 README 的「數據」和「已知局限」兩節。

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# 讓圖表能顯示中文（按你的系統保留一個能用的字體即可）
plt.rcParams["font.sans-serif"] = ["Microsoft JhengHei", "PingFang TC", "Arial Unicode MS", "SimHei"]
plt.rcParams["axes.unicode_minus"] = False

DATA = Path("..") / "data"
hfq = pd.read_parquet(DATA / "processed" / "daily_hfq.parquet")   # 後復權
raw = pd.read_parquet(DATA / "processed" / "daily_none.parquet")  # 不復權
cal = pd.read_csv(DATA / "raw" / "trade_calendar.csv", parse_dates=["trade_date"])
universe = pd.read_csv(DATA / "raw" / "universe_csi800.csv", dtype={"code": str})

hfq.head()

## 任務 1：基本概況

1. 數據一共多少行、多少隻股票、日期範圍是多少？
2. 每個交易日有多少隻股票有數據？畫成折線圖。
3. 股票池裡有沒有股票完全沒下載到數據？

**思考**：第 2 題的曲線在早年明顯偏低，為什麼？用「今天的中證 800 成分股」去研究 2015 年的市場，會有什麼問題？（關鍵詞：倖存者偏差）

> 提示：`hfq.groupby("date")["code"].nunique()`；第 3 題可以用集合差 `set(a) - set(b)`

## 任務 2：長表轉寬表，計算日收益

量化研究最常用的數據形狀是**寬表**：行是日期，列是股票代碼。後面算因子、算 IC 基本都在寬表上做。

1. 把後復權收盤價轉成寬表 `close`（date × code）。
2. 計算日收益率矩陣 `ret`。
3. 隨機挑 3 隻股票，畫出累計收益曲線 `(1 + ret).cumprod()`。

**思考**：某隻股票停牌 5 天後復牌，復牌當天的收益應該等於多少？你的 `ret` 在這一天給出的是什麼？如果兩者不一樣，怎麼修正？

> 提示：`hfq.pivot(index="date", columns="code", values="close")`。
> 不同 pandas 版本的 `pct_change` 遇到缺失值的默認行為不同，建議寫成 `close.pct_change(fill_method=None)`，然後**自己找一隻停牌過的股票**檢查停牌前後的值。

## 任務 3：理解復權

1. 取貴州茅台（600519）的後復權收盤價和不復權收盤價，各自除以第一天的值，畫在同一張圖上。
2. 分別用兩套價格算日收益率，找出兩者相差最大的幾天。查一下那幾天發生了什麼（提示：分紅除息）。
3. 用 `後復權收盤價 / 不復權收盤價` 算出**復權因子**，畫出來。它在什麼時候跳變？

**思考（面試常問）**：
- 為什麼計算收益要用後復權價格，而不是不復權價格？
- 前復權（qfq）也能算出正確的收益率，但今天下載和一年後下載的「歷史價格」會不一樣。為什麼？這對回測的可重複性有什麼影響？

## 任務 4：停牌

1. 對每隻股票，統計「從它第一次出現在數據裡開始，有多少個交易日沒有數據」。
2. 列出缺失天數最多的 5 隻股票，查一下原因。

**思考**：回測時停牌的股票能買賣嗎？如果策略某天選中了一隻停牌股票，應該怎麼處理？

> 提示：交易日曆 `cal` 裡可能包含未來日期，先篩到數據的最後一天。可以用任務 2 的寬表：`close.notna()` 是「有數據」的布爾矩陣。

## 任務 5：漲跌停與異常值

1. 用**不復權**數據的 `pct_chg` 列（單位是 %），找出單日漲跌幅絕對值超過 21% 的記錄。
2. 逐條判斷：哪些是合理的，哪些可能是數據錯誤？
3. 寫一個函數 `limit_pct(code, date)`，返回這隻股票在這一天的漲跌幅限制。

已知規則（其餘自己查交易所規則補全）：
- 主板 ±10%
- 科創板（688 開頭）±20%
- 創業板（300 / 301 開頭）在 2020-08-24 註冊制改革後 ±20%，之前 ±10%
- 註冊制下的新股，上市後前幾個交易日不設漲跌幅限制
- ST 股另有規定，而且近年調整過，請自己查

**為什麼要做這個**：漲停時通常買不進，跌停時賣不出。第 5 步回測時要用這個函數排除無法成交的交易，否則回測收益會虛高。

## 任務 6：寫結論、提交

把以下內容寫進 README：
- 數據來源、時間範圍、股票池、股票數量
- 你的處理決定（收益用哪種價格算；停牌日怎麼處理）
- 已知局限（倖存者偏差、未納入北交所……）

然後在終端運行：

```bash
git add .
git commit -m "data: add data check notebook and document limitations"
git push
```

## 自測：面試被問到時，你能回答嗎？

1. 什麼是倖存者偏差？在你的數據裡具體表現在哪裡？打算怎麼解決？
2. 為什麼算收益用後復權？前復權有什麼問題？
3. 停牌期間的收益記為 0 還是缺失值？兩種做法對後面計算 IC 有什麼不同影響？